# Hierarchical Relaxation Labeling (HRL) with Cantor-Set Adaptive Subdivision
### Rotated Ellipse Primitive Extraction (Man-Made Geometry vs. Organic Noise)

This updated notebook incorporates the full rotation angle ($	heta$) and anisotropic semi-axes ($a, b$) into the Cantor-HRL constraint engine, ensuring precise fitting of rotated man-made geometric primitives while rejecting fractal organic clutter.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math

# Set random seed for reproducibility
np.random.seed(42)

## 1. Synthetic Scene Generation: Rotated Ellipse vs. Organic Noise
We construct a 2D point cloud containing a rotated man-made ellipse (anisotropic axes $a 
eq b$, rotation angle $	heta = \pi/6$) embedded in heavy random scatter.

In [ ]:
def generate_synthetic_scene(n_noise=150, ellipse_params=(50, 50, 32, 14, np.pi/6), noise_bounds=(0, 100)):
    xc, yc, a, b, theta = ellipse_params
    
    # Generate clean points along the rotated ellipse perimeter
    angles = np.linspace(0, 2 * np.pi, 70)
    cos_t, sin_t = np.cos(theta), np.sin(theta)
    
    ex_local = a * np.cos(angles)
    ey_local = b * np.sin(angles)
    
    # Apply rotation and translation
    ex = xc + ex_local * cos_t - ey_local * sin_t
    ey = yc + ex_local * sin_t + ey_local * cos_t
    
    # Add slight sensor noise
    ex += np.random.normal(0, 0.8, size=ex.shape)
    ey += np.random.normal(0, 0.8, size=ey.shape)
    man_made_points = np.column_stack((ex, ey))
    
    # Generate organic noise points
    nx = np.random.uniform(noise_bounds[0], noise_bounds[1], n_noise)
    ny = np.random.uniform(noise_bounds[0], noise_bounds[1], n_noise)
    organic_points = np.column_stack((nx, ny))
    
    scene_points = np.vstack((man_made_points, organic_points))
    return scene_points, man_made_points, organic_points

true_params = (50, 50, 32, 14, np.pi/6)
scene_points, true_ellipse_pts, organic_pts = generate_synthetic_scene(ellipse_params=true_params)

plt.figure(figsize=(8, 8))
plt.scatter(organic_pts[:, 0], organic_pts[:, 1], c='forestgreen', alpha=0.4, label='Organic Noise')
plt.scatter(true_ellipse_pts[:, 0], true_ellipse_pts[:, 1], c='dodgerblue', s=30, label='True Rotated Ellipse')
plt.title('Synthetic Scene: Rotated Ellipse vs. Nature')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.axis('equal')
plt.show()

## 2. Rotated HRL Compatibility & Cantor Subdivision Engine
We evaluate ellipse hypotheses using the inverse rotation transformation to correctly measure algebraic residuals for anisotropic and rotated geometries.

In [ ]:
class RotatedCantorHRLConstraintEngine:
    def __init__(self, points):
        self.points = points
        
    def evaluate_compatibility(self, label_params):
        xc, yc, a, b, theta = label_params
        if a <= 3 or b <= 3 or a > 70 or b > 70:
            return 0.0
            
        cos_t, sin_t = np.cos(theta), np.sin(theta)
        dx = self.points[:, 0] - xc
        dy = self.points[:, 1] - yc
        
        # Rotate points into the ellipse's canonical frame
        x_prime =  dx * cos_t + dy * sin_t
        y_prime = -dx * sin_t + dy * cos_t
        
        # Ellipse algebraic equation check
        vals = (x_prime**2) / (a**2) + (y_prime**2) / (b**2)
        residuals = np.abs(vals - 1.0)
        
        support_count = np.sum(residuals < 0.25)
        mean_residual = np.mean(np.minimum(residuals, 1.0))
        score = support_count * np.exp(-1.8 * mean_residual)
        return float(score)

## 3. Multi-Scale Hierarchical Optimization with Cantor Subdivision

In [ ]:
engine = RotatedCantorHRLConstraintEngine(scene_points)

# Coarse initial search bounds for [xc, yc, a, b, theta]
current_bounds = [
    (35.0, 65.0), # xc
    (35.0, 65.0), # yc
    (15.0, 45.0), # a
    (5.0,  25.0), # b
    (0.0,  np.pi) # theta
]

iterations = 4
best_hypothesis = None
best_score = -1

print("Starting Rotated Cantor-HRL Optimization...")
for it in range(iterations):
    xc_vals = np.linspace(current_bounds[0][0], current_bounds[0][1], 6)
    yc_vals = np.linspace(current_bounds[1][0], current_bounds[1][1], 6)
    a_vals  = np.linspace(current_bounds[2][0], current_bounds[2][1], 5)
    b_vals  = np.linspace(current_bounds[3][0], current_bounds[3][1], 5)
    th_vals = np.linspace(current_bounds[4][0], current_bounds[4][1], 6)
    
    local_best_score = -1
    local_best_params = None
    
    for xc in xc_vals:
        for yc in yc_vals:
            for a in a_vals:
                for b in b_vals:
                    for th in th_vals:
                        score = engine.evaluate_compatibility((xc, yc, a, b, th))
                        if score > local_best_score:
                            local_best_score = score
                            local_best_params = (xc, yc, a, b, th)
                            
    if local_best_score > best_score:
        best_score = local_best_score
        best_hypothesis = local_best_params
        
    print(f"Iteration {it+1}: Best Score = {local_best_score:.2f} | Params (xc,yc,a,b,theta) = {np.round(local_best_params, 3)}")
    
    # Cantor-set interval refinement (shrinking bounds around current optimum)
    span = 0.4 / (it + 1)
    current_bounds = [
        (max(0, local_best_params[0] - 15*span), min(100, local_best_params[0] + 15*span)),
        (max(0, local_best_params[1] - 15*span), min(100, local_best_params[1] + 15*span)),
        (max(5, local_best_params[2] - 10*span), min(50,  local_best_params[2] + 10*span)),
        (max(3, local_best_params[3] - 8*span),  min(30,  local_best_params[3] + 8*span)),
        (max(0, local_best_params[4] - 0.5*span), min(np.pi, local_best_params[4] + 0.5*span))
    ]

print("\nOptimal Fitted Rotated Ellipse Parameters:", np.round(best_hypothesis, 4))

## 4. Visualization of Fitted Rotated Ellipse

In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(scene_points[:, 0], scene_points[:, 1], c='lightgray', s=15, label='Scene Clutter')
plt.scatter(true_ellipse_pts[:, 0], true_ellipse_pts[:, 1], c='dodgerblue', alpha=0.5, label='True Target')

if best_hypothesis:
    xc, yc, a, b, theta = best_hypothesis
    angles = np.linspace(0, 2 * np.pi, 150)
    cos_t, sin_t = np.cos(theta), np.sin(theta)
    
    ex_l = a * np.cos(angles)
    ey_l = b * np.sin(angles)
    fx = xc + ex_l * cos_t - ey_l * sin_t
    fy = yc + ex_l * sin_t + ey_l * cos_t
    
    plt.plot(fx, fy, 'r-', linewidth=3, label='HRL Cantor-Fitted Rotated Ellipse')
    plt.scatter([xc], [yc], c='red', marker='x', s=100, label='Estimated Center')

plt.title('Cantor-HRL Rotated Ellipse Extraction & Noise Rejection')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.axis('equal')
plt.show()